In [67]:
# File:     tensor_coord_transform_testing.ipynb

# Author:   Srinikkesh Senthilkumar
# Date:     November 7, 2026
# License:  MIT License (see LICENSE in root directory)
# Copyright (c) 2026 Srinikkesh Senthilkumar

# Summary of File:

#   This file is for experimenting with tensor coordinate transformations

In [68]:
import numpy as np 
from skimage.measure import marching_cubes
import trimesh
import pyvista as pv

In [69]:
# Initial Coordinates
resolution = 60
size = 25

bound_lower = -20
bound_higher = 20

x = np.linspace(bound_lower, bound_higher, resolution)
y = np.linspace(bound_lower, bound_higher, resolution)
z = np.linspace(bound_lower, bound_higher, resolution)
X, Y, Z = np.meshgrid(x, y, z)

coords = np.stack([X, Y, Z])

3x3 Deformation Gradient Tensor F is used to stretch and rotate the coordinate space to deform the implicit function

In [70]:
F = np.array([
    [2.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 3.0]
])

F_inv = np.linalg.inv(F)

In [71]:
coords_swapped = np.moveaxis(coords, 0, -1)
coords_original = np.einsum('ij, ...j -> ...i', F_inv, coords_swapped)
X_orig = coords_original[..., 0]
Y_orig = coords_original[..., 1]
Z_orig = coords_original[..., 2]


In [72]:
# Gyroid Implicit Function
field = np.cos(X_orig)*np.sin(Y_orig) + np.cos(Y_orig)*np.sin(Z_orig) + np.cos(Z_orig)*np.sin(X_orig)

In [73]:
# Meshing
# Extract sufaces
verts, faces, normals, values = marching_cubes(field, level=0.0)

# Scale to physical dimensions (mm)
verts = verts / resolution * size

mesh = trimesh.Trimesh(vertices=verts, faces=faces, normals=normals)

In [74]:
pv_mesh = pv.wrap(mesh)

plotter = pv.Plotter()
plotter.add_mesh(pv_mesh, color="lightblue", show_edges=True, smooth_shading=False)
plotter.add_axes() # type: ignore
plotter.show()

Widget(value='<iframe src="http://localhost:43371/index.html?ui=P_0x7f0b1abf20d0_7&reconnect=auto" class="pyvi…